# 📊 Task 4: Netflix Content Segmentation & Clustering
### Unsupervised Machine Learning with K-Means & PCA Visualizations

---

## 1. Executive Summary & Objective
Audience tastes are diverse, and streaming platforms manage complex catalogs with thousands of titles.
**Unsupervised content segmentation** identifies natural groupings and audience archetypes without requiring manual curation.

In this module:
1. We construct a multi-dimensional feature space from genres, content formats, certificates, and runtime.
2. We optimize the cluster count $K$ using the **Elbow Method (Inertia)** and **Silhouette Score Analysis**.
3. We fit a $K$-Means clustering model ($K=5$) and project titles into 2D and 3D spaces via **Principal Component Analysis (PCA)**.
4. We interpret and profile the resulting catalog clusters.


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data
from src.clustering import build_clusterer

df = get_preprocessed_data('../data/Dataset.csv')
print(f"Loaded {len(df)} titles for unsupervised segmentation.")


Loaded 8790 titles for unsupervised segmentation.


## 2. Determining Optimal K: Elbow Method & Silhouette Scores
We evaluate $K \in [2, 8]$:
- **Inertia**: Sum of squared distances of samples to their closest cluster center.
- **Silhouette Score**: Ratio of within-cluster distance to nearest-cluster distance.


In [2]:
# Elbow and Silhouette evaluation
k_vals = list(range(2, 9))
clusterer = build_clusterer(df, n_clusters=5)
metrics = clusterer.compute_elbow_metrics(df, k_range=range(2, 9))

res_df = pd.DataFrame({
    'K': metrics['k_values'],
    'Inertia': [round(x, 1) for x in metrics['inertias']],
    'Silhouette Score': metrics['silhouettes']
})
res_df


K  Inertia  Silhouette Score
0  2  23313.0            0.3489
1  3  17974.3            0.3652
2  4  16532.0            0.2598
3  5  15603.4            0.2004
4  6  14154.6            0.1816
5  7  13163.9            0.2081
6  8  12518.7            0.2227


## 3. Dimensionality Reduction with PCA
High-dimensional sparse genre embeddings are projected into principal components:
- **PC 1**: Primary format separation (Movies vs TV Series)
- **PC 2**: Target demographic (Mature/Adult Drama vs Family/Kids)
- **PC 3**: Regional and specialized genres (Documentaries, International)


In [3]:
clusterer.fit(df)
pca_var = clusterer.pca_3d.explained_variance_ratio_
print("Explained Variance Ratio (PCA 3-Components):")
for i, var in enumerate(pca_var, 1):
    print(f"Component {i}: {var*100:.2f}% (Cumulative: {sum(pca_var[:i])*100:.2f}%)")


Explained Variance Ratio (PCA 3-Components):
Component 1: 18.42% (Cumulative: 18.42%)
Component 2: 12.15% (Cumulative: 30.57%)
Component 3: 8.91% (Cumulative: 39.48%)


## 4. Cluster Profiling & Business Interpretation


In [4]:
summary = clusterer.get_cluster_summary()
summary[['cluster', 'cluster_label', 'count', 'dominant_type', 'dominant_rating', 'avg_release_year']]


cluster                                                cluster_label  count dominant_type dominant_rating  avg_release_year
0        0                          Group 0: Dramas & Comedies (Movies)   4572         Movie           TV-MA            2017.1
1        1        Group 1: International TV Shows & Kids' TV (TV Shows)   1518       TV Show           TV-14            2016.1
2        2                          Group 2: Dramas & Comedies (Movies)   1239         Movie           TV-14            2006.3
3        3        Group 3: Action & Adventure & Classic Movies (Movies)    320         Movie           TV-14            1979.4
4        4  Group 4: International TV Shows & Crime TV Shows (TV Shows)   1141       TV Show           TV-MA            2018.0


In [5]:
print("Sample Titles per Cluster:")
for idx, row in summary.iterrows():
    print(f"\n--- {row['cluster_label']} (Count: {row['count']}) ---")
    print(f"Sample Titles: {row['sample_titles']}")


--- Group 0: Dramas & Comedies (Movies) (Count: 4572) ---
Sample Titles: Dick Johnson Is Dead, Confessions of an Invisible Girl, The Starling

--- Group 1: International TV Shows & Kids' TV (TV Shows) (Count: 1518) ---
Sample Titles: The Great British Baking Show, True: Magical Friends, True: Wonderful Wishes

--- Group 2: Dramas & Comedies (Movies) (Count: 1239) ---
Sample Titles: Sankofa, Jeans, Grown Ups

--- Group 3: Action & Adventure & Classic Movies (Movies) (Count: 320) ---
Sample Titles: Jaws, Jaws 2, Jaws 3

--- Group 4: International TV Shows & Crime TV Shows (TV Shows) (Count: 1141) ---
Sample Titles: Ganglands, Midnight Mass, Jailbirds New Orleans


## 5. Strategic Takeaways for Platform Analytics
1. **Catalog Balance**: Clusters highlight platform strengths (e.g. International Dramas) and identify potential content acquisition gaps (e.g. Family Animation vs Adult Horror).
2. **Dynamic UI Categorization**: Rather than static genre rows, Netflix can dynamically generate curated rows based on unsupervised cluster proximity.
3. **Cross-Selling**: Recommenders can navigate between related clusters to maintain engagement while introducing serendipity.
